# NY Hospital Discharge Analysis — Data Cleaning
**Dataset:** Hospital Inpatient Discharges (SPARCS De-Identified) 2021  
**Source:** NY State Department of Health via Kaggle  
**Records:** 2,052,711 | **Original columns:** 32 | **Final columns:** 14

This notebook covers data preparation before Power BI analysis:
1. Load and inspect raw data
2. Analyse missing values
3. Select relevant columns
4. Fix Length of Stay (remove '120+' strings)
5. Convert Total Charges and Total Costs to numeric
6. Create derived columns
7. Drop index column and export

## 1. Load and inspect raw data

In [ ]:
import pandas as pd

df = pd.read_csv('Hospital_Inpatient_Discharges__SPARCS_De-Identified___2021.csv')

In [ ]:
df.info()

In [ ]:
df.head()

## 2. Analyse missing values

In [ ]:
round(df.isnull().sum() / len(df), 2) * 100

**Key observations:**
- `CCSR Procedure Code` and `CCSR Procedure Description`: 27% missing — excluded
- `Payment Typology 2` and `Payment Typology 3`: 51% and 84% missing — excluded
- `Birth Weight`: 90% missing — excluded
- `Total Charges` and `Total Costs`: 0% missing ✅

## 3. Select relevant columns
Reduce from 32 to 12 core columns.

In [ ]:
df_compact = df[[
    'Facility Name',
    'Age Group',
    'Gender',
    'Race',
    'Length of Stay',
    'Type of Admission',
    'CCSR Diagnosis Description',
    'APR MDC Description',
    'APR Risk of Mortality',
    'Payment Typology 1',
    'Total Charges',
    'Total Costs'
]].copy()

In [ ]:
df_compact.info()

## 4. Fix Length of Stay — remove '120+' string values

In [ ]:
# Check unique values — notice '120+' string mixed with integers
df_compact['Length of Stay'].unique()

In [ ]:
# Remove '+' and convert to integer
df_compact['Length of Stay'] = (
    df_compact['Length of Stay']
    .astype(str)
    .str.replace('+', '', regex=False)
    .str.strip()
    .astype(int)
)

In [ ]:
df_compact.info()

## 5. Convert Total Charges and Total Costs to numeric
Both columns are stored as strings with commas — need to strip and convert to float.

In [ ]:
# Verify current dtype — shows 'object' (string)
print(df_compact['Total Charges'].head())
print(df_compact['Total Charges'].dtype)

In [ ]:
df_compact['Total Charges'] = pd.to_numeric(
    df_compact['Total Charges'].str.strip().str.replace(',', '', regex=False),
    errors='coerce'
).fillna(0)

df_compact['Total Costs'] = pd.to_numeric(
    df_compact['Total Costs'].str.strip().str.replace(',', '', regex=False),
    errors='coerce'
).fillna(0)

In [ ]:
# Verify conversion worked
df_compact['Total Charges']

## 6. Create derived columns

In [ ]:
df_compact['Total Charges per Day'] = df_compact['Total Charges'] / df_compact['Length of Stay']
df_compact['Total Costs per Day'] = df_compact['Total Costs'] / df_compact['Length of Stay']

In [ ]:
df_compact

## 7. Drop index column and export

In [ ]:
if 'Unnamed: 0' in df_compact.columns:
    df_compact = df_compact.drop(columns=['Unnamed: 0'])

In [ ]:
df_compact.to_csv('Hospital_Inpatient_Discharges__De-Identified___2021.csv', index=False)
print('Done. Ready for Power BI.')